# Colab 실행 안내

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glasslego/2026-ml-study/blob/main/notebooks/study_02_cnn_rnn/02_rnn_lstm_essentials.ipynb)

이전 노트북 (`01_cnn_essentials.ipynb`) 에서 CNN 의 핵심 — locality 와 weight sharing — 을 봤다.
이번에는 **시간 (시퀀스)** 이라는 또 다른 차원을 다루는 **RNN / LSTM** 의 본질을 본다.

흐름:

1. 시퀀스 데이터와 RNN 의 동기 (왜 MLP 만으로 안 되나)
2. RNN 의 수식
3. numpy 로 RNN 한 timestep forward 직접 구현
4. 시간 펼치기 (unrolling) 와 BPTT
5. Vanishing / Exploding Gradient 시연
6. LSTM 의 등장과 게이트 수식
7. numpy 로 LSTM 한 cell forward
8. PyTorch `nn.RNN`, `nn.LSTM` 사용 + numpy 결과 일치 검증
9. 작은 char-level 언어모델 학습
10. RNN/LSTM 의 한계 → Transformer 로 가는 동기
11. 흔한 함정 + gradient clipping 시연

> 핵심 메시지: **RNN 의 weight sharing 은 "시간 축" 에서 일어난다.**
> 모든 timestep 이 같은 $W_{xh}, W_{hh}$ 를 공유한다는 점이
> CNN 이 모든 위치에서 같은 커널을 공유하는 것과 정확히 평행한 발상이다.

In [ ]:
# 공통 실행 환경 준비 (Colab / Local 자동 분기)
from pathlib import Path
import subprocess
import sys
import os

REPO_URL = 'https://github.com/glasslego/2026-ml-study.git'
REPO_NAME = '2026-ml-study'
NOTEBOOK_SUBDIR = 'notebooks/study_02_cnn_rnn'

if 'google.colab' in sys.modules:
    clone_root = Path('/content') / REPO_NAME
    if not clone_root.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(clone_root)], check=True)
    target = clone_root / NOTEBOOK_SUBDIR
else:
    target = None
    for c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (c / NOTEBOOK_SUBDIR).exists():
            target = c / NOTEBOOK_SUBDIR
            break
    if target is None:
        raise FileNotFoundError(f'{NOTEBOOK_SUBDIR} 를 찾지 못했습니다.')

os.chdir(target)
print(f'작업 디렉토리: {target}')

# torch 가 없으면 설치 (Colab 에는 기본 설치되어 있음)
try:
    import torch  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch'], check=True)
    import torch  # noqa: F401


In [ ]:
# 핵심 import 및 재현성 설정
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 20190501
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"torch version: {torch.__version__}")


# 1. 시퀀스 데이터와 RNN 의 동기

MLP / CNN 은 **고정 크기** 입력을 가정한다 (이미지 28x28 등).
하지만 다음 데이터들은 **가변 길이** 다:

- 자연어 문장 ("나는 학교에 갔다" 5 토큰 vs "오늘 점심 뭐 먹지" 5 토큰)
- 음성 파형 (시간 길이가 발화마다 다름)
- 주식 가격 (날짜 수가 종목마다 다름)
- 동영상 프레임

**RNN 의 핵심 아이디어**:
> "이전 상태 + 현재 입력 → 새 상태" 라는 **재귀 (recursion)** 함수 한 개로
> 임의 길이 시퀀스를 처리한다.

그리고 그 함수 (= 가중치) 는 **모든 timestep 에서 공유** 된다.
이것이 RNN 의 inductive bias — **temporal weight sharing** — 이다.

# 2. RNN 수식

가장 기본 (vanilla) RNN 의 한 timestep:

$$ h_t = \tanh(W_{xh}\, x_t + W_{hh}\, h_{t-1} + b_h) $$
$$ y_t = W_{hy}\, h_t + b_y $$

여기서:
- $x_t \in \mathbb{R}^{D_x}$ : timestep $t$ 의 입력
- $h_t \in \mathbb{R}^{D_h}$ : timestep $t$ 의 은닉 상태
- $W_{xh} \in \mathbb{R}^{D_h \times D_x}$ : 입력 → 은닉
- $W_{hh} \in \mathbb{R}^{D_h \times D_h}$ : 은닉 → 은닉 (재귀 가중치)
- $W_{hy} \in \mathbb{R}^{D_y \times D_h}$ : 은닉 → 출력
- $b_h, b_y$ : 편향

**중요한 점**: $W_{xh}, W_{hh}, W_{hy}, b_h, b_y$ 는 모든 timestep $t$ 에 대해 **같은 값**.
아무리 시퀀스가 길어져도 파라미터 수는 증가하지 않는다.

In [ ]:
def rnn_step_numpy(x_t, h_prev, Wxh, Whh, b_h):
    """RNN 한 timestep forward (numpy).

    Args:
        x_t:    (D_x,)
        h_prev: (D_h,)
        Wxh:    (D_h, D_x)
        Whh:    (D_h, D_h)
        b_h:    (D_h,)

    Returns:
        h_t: (D_h,)
    """
    pre = Wxh @ x_t + Whh @ h_prev + b_h
    h_t = np.tanh(pre)
    return h_t


# 작은 예시: D_x=4, D_h=3
D_x, D_h = 4, 3
rng = np.random.default_rng(0)
Wxh = rng.standard_normal((D_h, D_x)).astype(np.float32) * 0.5
Whh = rng.standard_normal((D_h, D_h)).astype(np.float32) * 0.5
b_h = np.zeros(D_h, dtype=np.float32)

x_t = rng.standard_normal(D_x).astype(np.float32)
h_prev = np.zeros(D_h, dtype=np.float32)

h_t = rnn_step_numpy(x_t, h_prev, Wxh, Whh, b_h)
print(f"x_t shape: {x_t.shape}")
print(f"h_prev shape: {h_prev.shape}")
print(f"h_t  : {h_t}")


# 3. 시간 펼치기 (Unrolling) 와 BPTT

길이 $T$ 시퀀스를 처리하는 RNN 은 **같은 RNN cell 을 $T$ 번 직렬로 쌓은 것** 과 같다.
이를 **시간 축으로 펼친다 (unroll)** 고 부른다.

```
x_1 → [RNN] → h_1 → [RNN] → h_2 → ... → [RNN] → h_T
                ↑              ↑                   ↑
              x_2            x_3                 x_T
```

각 [RNN] 박스의 가중치는 **모두 같다** ($W_{xh}, W_{hh}$ 한 벌).

**BPTT (Backprop Through Time)**:
- 펼친 그래프에 대해 일반적인 backprop 을 적용한 것.
- $\dfrac{\partial L}{\partial W_{hh}}$ 는 모든 timestep 의 기여를 **합** 한다 (weight sharing 이라서).
- 그래서 긴 시퀀스에서는 chain rule 곱이 매우 길어진다 — 다음 절의 vanishing/exploding 문제로 이어진다.

In [ ]:
def rnn_forward_numpy(xs, h0, Wxh, Whh, b_h):
    """T 개 timestep forward 펼치기.

    Args:
        xs:  (T, D_x)
        h0:  (D_h,)

    Returns:
        H:  (T, D_h)  — 모든 timestep 의 hidden state
        h_T: (D_h,)   — 마지막 hidden state
    """
    T = xs.shape[0]
    D_h = h0.shape[0]
    H = np.zeros((T, D_h), dtype=xs.dtype)
    h = h0.copy()
    for t in range(T):
        h = rnn_step_numpy(xs[t], h, Wxh, Whh, b_h)
        H[t] = h
    return H, h


T = 5
xs = rng.standard_normal((T, D_x)).astype(np.float32)
H, h_T = rnn_forward_numpy(xs, h0=np.zeros(D_h, dtype=np.float32),
                            Wxh=Wxh, Whh=Whh, b_h=b_h)
print(f"입력 시퀀스 shape: {xs.shape}  (T={T}, D_x={D_x})")
print(f"hidden 시퀀스 shape: {H.shape}  (T={T}, D_h={D_h})")
print(f"마지막 h_T: {h_T}")


# 4. Vanishing / Exploding Gradient

BPTT 에서 hidden state 의 기울기는 chain rule 로 거슬러 올라간다:

$$ \dfrac{\partial h_t}{\partial h_{t-1}} = W_{hh}^\top \cdot \mathrm{diag}(\tanh'(\cdot)) $$

따라서 시퀀스 길이 $T$ 에 걸친 누적 기울기는 대략:

$$ \dfrac{\partial h_T}{\partial h_0} = \prod_{t=1}^{T} \dfrac{\partial h_t}{\partial h_{t-1}} \approx \prod_{t=1}^{T} W_{hh}^\top \cdot \mathrm{diag}(\tanh'(\cdot)) $$

이 곱의 norm 은 대략 $\|W_{hh}\|^T \cdot \prod \|\tanh'\|$ 처럼 거동한다.
$\tanh'$ 는 항상 $\le 1$ 이므로:

- **$W_{hh}$ 의 spectral radius < 1** → 기울기 0 으로 **vanishing** (먼 과거 신호를 학습 못 함)
- **spectral radius > 1** → 기울기가 **exploding** (학습 불안정 / NaN)

이것이 vanilla RNN 이 긴 시퀀스 (예: 50+ token) 에서 잘 학습되지 않는 본질적 이유다.

In [ ]:
# Vanishing / Exploding 시연 — T 가 커질수록 dh_T / dh_0 의 norm 이 어떻게 변하는가
# 누적 jacobian: J_T ≈ ∏ (W_hh^T · diag(tanh')) → norm ≈ (sr(W_hh) · |tanh'|)^T
# tanh' 는 항상 ≤ 1 이고 hidden 에 따라 다른데, 시뮬레이션에서는 평균값 ~0.5 로 둔다.
# 따라서 "실효 spectral radius" = sr(W_hh) · 0.5 가 1 보다 큰지 작은지가 갈림길.

D_h = 16
TANH_PRIME = 0.5     # tanh' 의 평균값 가정 (실제로는 hidden 에 의존)
rng = np.random.default_rng(0)
W = rng.standard_normal((D_h, D_h))
U, _, Vt = np.linalg.svd(W)

def with_spectral_radius(sr):
    """spectral radius 가 정확히 sr 인 행렬을 만든다."""
    return U @ np.diag(np.full(D_h, sr)) @ Vt

# vanishing 쪽: sr · 0.5 = 0.25 (< 1)
# exploding 쪽: sr · 0.5 = 1.50 (> 1)  ← 폭주가 실제로 보이려면 sr > 2 가 필요!
W_small = with_spectral_radius(0.5)
W_big   = with_spectral_radius(3.0)

def jacobian_norm_after_T_steps(W_hh, T, tanh_prime=TANH_PRIME):
    J = np.eye(D_h)
    for _ in range(T):
        J = (W_hh.T * tanh_prime) @ J
    return np.linalg.norm(J)

print(f"실효 spectral radius (= sr·tanh'):  small = {0.5*TANH_PRIME:.2f},  big = {3.0*TANH_PRIME:.2f}")
print()
print(f"{'T':>4}  {'small (sr=0.5)':>18}  {'big (sr=3.0)':>18}")
for T in [1, 5, 10, 20, 50, 100]:
    n_small = jacobian_norm_after_T_steps(W_small, T)
    n_big   = jacobian_norm_after_T_steps(W_big, T)
    print(f"{T:>4}  {n_small:>18.3e}  {n_big:>18.3e}")

print("\n→ small: 실효 sr < 1 → 기하급수적 0 수렴 (vanishing)")
print("→ big  : 실효 sr > 1 → 기하급수적 폭주 (exploding)")
print("→ 둘 다 긴 시퀀스에서 BPTT 학습을 어렵게 만든다.")


# 5. LSTM 의 등장

LSTM (Long Short-Term Memory, Hochreiter & Schmidhuber 1997) 은
**게이트 (gate)** 를 도입해서 "무엇을 잊고, 무엇을 기억할지"를 데이터에서 학습하게 만들었다.

각 게이트는 시그모이드 출력 (0~1) 으로, **원소별 곱** 으로 정보 흐름을 조절한다.

수식 (한 timestep, $[h_{t-1}, x_t]$ 는 두 벡터를 이어붙인 것):

$$ f_t = \sigma(W_f\, [h_{t-1}, x_t] + b_f) \quad \text{(forget gate)} $$
$$ i_t = \sigma(W_i\, [h_{t-1}, x_t] + b_i) \quad \text{(input gate)} $$
$$ \tilde{C}_t = \tanh(W_C\, [h_{t-1}, x_t] + b_C) \quad \text{(candidate cell)} $$
$$ C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t \quad \text{(cell state update)} $$
$$ o_t = \sigma(W_o\, [h_{t-1}, x_t] + b_o) \quad \text{(output gate)} $$
$$ h_t = o_t \odot \tanh(C_t) $$

**핵심 직관 — 왜 vanishing 이 완화되나**:

- vanilla RNN 의 hidden 업데이트는 $h_t = \tanh(W h_{t-1} + ...)$ 로 **곱셈만** 으로 누적.
- LSTM 의 cell state 업데이트는 $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$ 로 **덧셈** 경로가 있다.
- $f_t \approx 1$ 이면 $C_t \approx C_{t-1}$ — gradient 가 거의 그대로 통과 (gradient highway).
- 곱셈 폭주/소실 없이 먼 과거의 정보를 길게 유지할 수 있다.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def lstm_step_numpy(x_t, h_prev, c_prev, W, b):
    """LSTM 한 timestep forward (numpy).

    구현 편의를 위해 4개 게이트의 가중치를 하나로 합쳐 둔다 (PyTorch 와 동일한 관습).

    Args:
        x_t:    (D_x,)
        h_prev: (D_h,)
        c_prev: (D_h,)
        W:      (4*D_h, D_x + D_h)   ← [W_i, W_f, W_g(C), W_o] 를 세로로 stack
        b:      (4*D_h,)

    Returns:
        h_t, c_t : 각 (D_h,)
    """
    D_h = h_prev.shape[0]
    z = np.concatenate([x_t, h_prev])           # (D_x + D_h,)
    pre = W @ z + b                              # (4*D_h,)

    # 게이트 4개를 잘라낸다 (입력 / 망각 / 후보 / 출력 순서 — PyTorch 관습과 동일)
    i_t = sigmoid(pre[0 * D_h : 1 * D_h])
    f_t = sigmoid(pre[1 * D_h : 2 * D_h])
    g_t = np.tanh(pre[2 * D_h : 3 * D_h])       # candidate cell
    o_t = sigmoid(pre[3 * D_h : 4 * D_h])

    c_t = f_t * c_prev + i_t * g_t              # cell state 의 덧셈 경로 — gradient highway
    h_t = o_t * np.tanh(c_t)
    return h_t, c_t


# 작은 예시
D_x, D_h = 4, 3
rng = np.random.default_rng(1)
W = rng.standard_normal((4 * D_h, D_x + D_h)).astype(np.float32) * 0.3
b = np.zeros(4 * D_h, dtype=np.float32)

x_t = rng.standard_normal(D_x).astype(np.float32)
h_prev = np.zeros(D_h, dtype=np.float32)
c_prev = np.zeros(D_h, dtype=np.float32)

h_t, c_t = lstm_step_numpy(x_t, h_prev, c_prev, W, b)
print(f"h_t: {h_t}")
print(f"c_t: {c_t}")


# 6. PyTorch `nn.RNN`, `nn.LSTM` 사용

**입력 모양 관습**:
- 기본: `(T, N, D)` — sequence-first (역사적 이유)
- `batch_first=True`: `(N, T, D)` — 더 직관적, 요즘 더 자주 쓴다

**hidden 초기화**:
- 명시적으로 `h_0 = torch.zeros(num_layers, N, D_h)` 를 넘겨도 되고
- 안 넘기면 자동으로 0 으로 초기화된다

**출력**:
- `output, h_n = rnn(x, h_0)`
- `output`: 모든 timestep 의 hidden — `(T, N, D_h)` 또는 `(N, T, D_h)`
- `h_n`: 마지막 timestep 의 hidden — `(num_layers, N, D_h)`
- LSTM 은 추가로 `c_n` 도 함께 반환 → `(output, (h_n, c_n))`

In [ ]:
# 우리 numpy RNN 과 PyTorch nn.RNN 의 결과가 일치하는지 검증
# (가중치를 손으로 복사해 넣어 둠)

D_x, D_h = 4, 3
T = 5

rng = np.random.default_rng(42)
Wxh_np = rng.standard_normal((D_h, D_x)).astype(np.float32) * 0.3
Whh_np = rng.standard_normal((D_h, D_h)).astype(np.float32) * 0.3
b_h_np = rng.standard_normal(D_h).astype(np.float32) * 0.1

xs_np = rng.standard_normal((T, D_x)).astype(np.float32)

# (a) numpy 결과
H_np, _ = rnn_forward_numpy(xs_np, h0=np.zeros(D_h, dtype=np.float32),
                             Wxh=Wxh_np, Whh=Whh_np, b_h=b_h_np)

# (b) PyTorch nn.RNN
# PyTorch 의 RNN 은 b_ih (입력 쪽 bias) 와 b_hh (hidden 쪽 bias) 두 벌의 bias 를 갖는다.
# 우리는 b_h 한 벌만 쓰므로 b_ih=0, b_hh=b_h 로 맞춘다.
rnn = nn.RNN(input_size=D_x, hidden_size=D_h, num_layers=1,
             nonlinearity='tanh', bias=True, batch_first=False)

with torch.no_grad():
    rnn.weight_ih_l0.copy_(torch.from_numpy(Wxh_np))
    rnn.weight_hh_l0.copy_(torch.from_numpy(Whh_np))
    rnn.bias_ih_l0.zero_()
    rnn.bias_hh_l0.copy_(torch.from_numpy(b_h_np))

xs_t = torch.from_numpy(xs_np).reshape(T, 1, D_x)   # (T, N=1, D_x)
h0_t = torch.zeros(1, 1, D_h)
H_t, h_n = rnn(xs_t, h0_t)
H_t = H_t.detach().numpy().reshape(T, D_h)

print("numpy:")
print(H_np)
print("\nPyTorch:")
print(H_t)

assert np.allclose(H_np, H_t, atol=1e-5), "결과가 다름!"
print("\nOK — numpy 결과와 PyTorch nn.RNN 결과가 정확히 일치한다.")


In [ ]:
# nn.LSTM 한 줄 사용 예 (batch_first=True — 더 직관적)
N, T, D_x, D_h = 8, 10, 5, 16

lstm = nn.LSTM(input_size=D_x, hidden_size=D_h, num_layers=1, batch_first=True)
x = torch.randn(N, T, D_x)            # (batch, time, feature)

# h_0, c_0 를 안 넘기면 자동으로 0 으로 초기화된다.
output, (h_n, c_n) = lstm(x)

print(f"input  : {tuple(x.shape)}        (N, T, D_x)")
print(f"output : {tuple(output.shape)}   (N, T, D_h)  ← 모든 timestep 의 hidden")
print(f"h_n    : {tuple(h_n.shape)}    (num_layers, N, D_h)  ← 마지막 timestep 의 hidden")
print(f"c_n    : {tuple(c_n.shape)}    (num_layers, N, D_h)  ← 마지막 cell state")

# LSTM 의 파라미터 수: 4개 게이트 * (D_x*D_h + D_h*D_h + 2*D_h)
expected = 4 * (D_x * D_h + D_h * D_h + 2 * D_h)
actual = sum(p.numel() for p in lstm.parameters())
print(f"\n파라미터 수 — 공식: {expected}, 실제: {actual}")


# 7. 작은 char-level 언어모델

LSTM 을 **다음 문자 예측** task 에 써 본다 (가장 작은 의미 있는 sequence task).

데이터: 짧은 문자열을 반복한 것 → 같은 패턴이 반복되므로 LSTM 이 빨리 학습한다.

흐름:
1. 문자 → 정수 인덱스 매핑
2. `nn.Embedding` 으로 인덱스를 vector 로
3. `nn.LSTM` 으로 시퀀스 처리
4. `nn.Linear` 로 vocab_size 차원 logits
5. `CrossEntropyLoss` 로 다음 문자 예측 loss

In [ ]:
import time

# (1) 데이터 — 짧은 문자열 반복
text = ("hello world! " * 200).strip()
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
vocab_size = len(chars)
print(f"문자 집합: {chars}  (vocab_size = {vocab_size})")
print(f"전체 길이: {len(text)} 문자")

data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
print(f"data shape: {data.shape}")

# (2) (T 길이) 입력 / (T 길이) target — target 은 입력을 한 칸 shift
T = 32
def get_batch(batch_size=16):
    starts = torch.randint(0, len(data) - T - 1, (batch_size,))
    x = torch.stack([data[s : s + T]     for s in starts])
    y = torch.stack([data[s + 1 : s + 1 + T] for s in starts])
    return x, y


# (3) 모델
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim=16, hidden=32):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, x, h_c=None):
        e = self.emb(x)                      # (N, T, emb_dim)
        out, h_c = self.lstm(e, h_c)         # (N, T, hidden)
        logits = self.fc(out)                # (N, T, vocab_size)
        return logits, h_c


torch.manual_seed(SEED)
model = CharLSTM(vocab_size)
opt = torch.optim.Adam(model.parameters(), lr=5e-3)
loss_fn = nn.CrossEntropyLoss()

# (4) 학습 루프
model.train()
t0 = time.time()
for step in range(300):
    x, y = get_batch(batch_size=32)
    opt.zero_grad()
    logits, _ = model(x)
    # CrossEntropyLoss 는 (N, C, ...) 모양을 기대 — flatten 해서 넣는다.
    loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
    loss.backward()
    opt.step()
    if step % 50 == 0:
        print(f"step {step:4d}  loss={loss.item():.4f}")

print(f"\n경과 시간: {time.time() - t0:.1f}s")

# (5) 샘플링 — "h" 부터 시작해 다음 문자를 30개 예측
model.eval()
with torch.no_grad():
    idx = torch.tensor([[stoi['h']]])   # (N=1, T=1)
    h_c = None
    out_chars = ['h']
    for _ in range(30):
        logits, h_c = model(idx, h_c)
        # 마지막 timestep 의 logits 에서 argmax (greedy)
        next_idx = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        out_chars.append(itos[int(next_idx)])
        idx = next_idx
    print("샘플 출력:", "".join(out_chars))


# 8. RNN/LSTM 의 한계 → Transformer 의 동기

LSTM 은 vanilla RNN 의 vanishing 문제를 크게 완화했지만, 여전히 두 가지 본질적 한계가 있다:

1. **순차 처리 → 병렬화 불가능**
   - $h_t$ 를 계산하려면 $h_{t-1}$ 이 필요. timestep 끼리 데이터 의존성이 있다.
   - GPU 의 대규모 병렬성을 제대로 활용 못 함 → **학습 속도가 느리다**.
   - 시퀀스 길이가 길수록 wall clock 시간이 선형으로 증가.

2. **긴 의존성은 여전히 약하다**
   - LSTM 의 cell state highway 가 도움은 되지만, 수백~수천 토큰 거리의 의존성은 여전히 학습이 어렵다.
   - 정보가 hidden state 라는 **고정 크기 병목** 을 통과해야 한다.

**Transformer 의 해법** (2017, "Attention is All You Need"):
- 순차 처리를 버리고 **모든 토큰이 모든 토큰을 한 번에 본다** (self-attention).
- 각 위치의 출력 = 다른 모든 위치의 가중합.
- 위치 정보는 별도로 (positional encoding) 주입.
- 결과: **병렬 학습 가능 + 장거리 의존성 직접 모델링**.

이 노트북 다음 폴더 (`study_03_transformer/`) 에서 attention → Transformer 로 자연스럽게 이어진다.

# 9. 흔한 함정

1. **hidden 초기화를 잊음**
   - PyTorch RNN/LSTM 은 자동으로 0 초기화해 주지만, **batch size 가 바뀌면 직접 다시 만들어야** 한다.
   - 시퀀스 별로 hidden 을 유지/초기화할지 명확히 정해야 한다.

2. **`batch_first` 헷갈림**
   - 기본은 `(T, N, D)`, `batch_first=True` 면 `(N, T, D)`.
   - DataLoader 의 collate 와 모양이 안 맞아서 dim 0/1 이 뒤집힌 채로 학습되는 경우가 매우 흔하다.

3. **가변 길이 시퀀스에서 padding 만 쓰고 끝**
   - 짧은 시퀀스에 0 padding 만 채워 넣으면 padding 도 hidden 업데이트에 영향을 준다.
   - 해결: `nn.utils.rnn.pack_padded_sequence` / `pad_packed_sequence` 또는 attention mask.

4. **Gradient clipping 안 씀**
   - 위에서 본 exploding gradient 는 LSTM 이라도 완전히 막지 못한다.
   - **`torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)`** 한 줄을
     `optimizer.step()` 직전에 넣는 것이 사실상 표준.

아래 셀에서 gradient clipping 을 시연한다.

In [ ]:
# Gradient clipping 시연 — 일부러 gradient 가 폭발하게 만들고, clip 으로 막는다.

torch.manual_seed(SEED)

# 큰 가중치로 vanilla RNN 만들어 exploding 유도
rnn = nn.RNN(input_size=8, hidden_size=64, num_layers=1, batch_first=True)
with torch.no_grad():
    # weight_hh 를 의도적으로 크게 만들어 spectral radius >> 1 로 만든다
    rnn.weight_hh_l0.mul_(5.0)

# 긴 시퀀스 (T=80) 를 하나 흘려보내고 backward 한다
x = torch.randn(2, 80, 8)
out, _ = rnn(x)
loss = out.pow(2).mean()   # 임의의 손실
loss.backward()

# clip 전 gradient norm
def total_grad_norm(params):
    norms = [p.grad.detach().norm() for p in params if p.grad is not None]
    return torch.norm(torch.stack(norms)).item()

before = total_grad_norm(rnn.parameters())
print(f"clip 전 total grad norm: {before:.2e}   (매우 크다 — exploding)")

# clip
clipped = torch.nn.utils.clip_grad_norm_(rnn.parameters(), max_norm=1.0)
print(f"clip 함수가 보고한 원래 norm: {clipped:.2e}")

after = total_grad_norm(rnn.parameters())
print(f"clip 후 total grad norm: {after:.4f}   (max_norm=1.0 이하로 제한됨)")

print("\n→ 학습 루프에서는 보통 다음과 같이 사용한다:")
print("    loss.backward()")
print("    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)")
print("    optimizer.step()")


# 정리

| 개념                  | 한 줄 요약                                                          |
|----------------------|---------------------------------------------------------------------|
| RNN 동기              | 가변 길이 입력을 "이전 상태 + 현재 입력 → 새 상태" 재귀로 처리         |
| Weight sharing        | 모든 timestep 이 같은 $W_{xh}, W_{hh}$ 를 공유                         |
| BPTT                  | 시간 펼친 그래프에서 일반 backprop                                    |
| Vanishing/Exploding   | $\prod W_{hh} \cdot \tanh'$ 의 norm 이 0 또는 무한대로                 |
| LSTM cell state       | $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$ — **덧셈** 경로가 highway |
| 모양 관습             | `(T, N, D)` 또는 `batch_first=True` 면 `(N, T, D)`                     |
| Gradient clipping     | `clip_grad_norm_(params, 1.0)` — 사실상 표준                          |
| 한계 → Transformer    | 순차 처리 → 병렬 불가, 장거리 의존성 약함 → self-attention 으로 해결    |

다음 단계: `study_03_transformer/` 에서 self-attention 의 수식과 직접 구현으로 이어진다.